# 03 — Modelos base

Este notebook documentará M1–M3 paso a paso. Ya completa **M1: preparación de experimentos** y **M2: evaluación temporal**.

Un **experimento** es una ejecución con datos, variables y parámetros definidos. Su registro permite saber exactamente cómo se obtuvo un resultado.

## 1. Configuración única

La semilla y las rutas se guardan en `configs/modeling.yaml`. Una **semilla** es un número que permite repetir los mismos pasos aleatorios.

In [ ]:
from src.evaluation.experiment import load_experiment_config, set_seed

config = load_experiment_config()
config

## 2. Comprobar la semilla

Al reiniciar la misma semilla, Python y NumPy deben producir los mismos números. Esto no garantiza que todo modelo sea idéntico, pero elimina una fuente común de variación.

In [ ]:
import random
import numpy as np

seed = config['experiment']['seed']
set_seed(seed)
primera_prueba = (random.random(), np.random.random())

set_seed(seed)
segunda_prueba = (random.random(), np.random.random())

assert primera_prueba == segunda_prueba
primera_prueba

## 3. Información mínima de cada ejecución

Cada ejecución registrará en MLflow:

- versión del código en Git;
- versión de los datos en DVC;
- objetivo y periodo evaluado;
- vista completa o sin texto compartido;
- variables de entrada;
- semilla;
- ejecución local o en Khipu;
- parámetros y métricas del modelo.

Una **métrica** es un número usado para evaluar un resultado. Por ejemplo, Macro-F1 para T1.

In [ ]:
from src.evaluation.experiment import build_run_record

registro_ejemplo = build_run_record(
    config=config,
    run_name='ejemplo-no-publicar',
    stage='M1',
    target='experiment_setup',
    split='not_applicable',
    view='not_applicable',
    features=[],
)
registro_ejemplo

## 4. Ejecuciones en Khipu

Los nodos SLURM no tienen acceso a internet. El trabajo guarda primero un registro JSON local. Al terminar, ese registro se publica en MLflow desde el nodo de acceso.

Esto separa dos acciones:

1. **Ejecutar:** calcular el resultado dentro de SLURM.
2. **Publicar:** enviar parámetros, métricas y artefactos pequeños a MLflow.

Los artefactos grandes, como embeddings o índices FAISS, se guardarán con DVC.

# M2 — Evaluación temporal

Una **división temporal** separa los datos por fecha. El modelo aprende con el pasado y se evalúa con datos posteriores. Esto representa mejor el uso real que una división aleatoria.

## 5. Periodos congelados

- **Ajuste:** enero de 2023 a septiembre de 2024. El modelo aprende aquí.
- **Calibración:** octubre a diciembre de 2024. Aquí se eligen parámetros y umbrales.
- **Validación temporal:** enero a junio de 2025. Mide el comportamiento posterior.

Un **umbral** convierte una probabilidad en una decisión. Por ejemplo, un umbral de 0.30 marca como positivo un caso con probabilidad de 0.35.

In [ ]:
import json
from src.evaluation.experiment import PROJECT_ROOT

ruta_contrato = PROJECT_ROOT / config['paths']['evaluation_contract']
evaluacion = json.loads(ruta_contrato.read_text(encoding='utf-8'))
evaluacion['splits']

## 6. Dos vistas de evaluación

La vista **completa** conserva todos los casos elegibles. La vista **sin texto compartido** excluye narrativas que ya aparecieron en periodos usados para aprender.

La segunda será la vista principal para elegir modelos porque reduce el beneficio artificial de memorizar plantillas. La vista completa también se conserva porque las plantillas existen en la operación real.

In [ ]:
import pandas as pd

pd.DataFrame(evaluacion['row_counts']).T

## 7. Métricas congeladas

- **Macro-F1 (T1):** calcula F1 por motivo y da el mismo peso a cada motivo.
- **Top-3 (T1):** revisa si el motivo correcto aparece entre tres sugerencias.
- **Average precision o precisión promedio (T2–T4):** resume la relación entre precisión y cobertura.
- **Precisión:** de los casos marcados positivos, cuántos eran positivos.
- **Cobertura o recall:** de los positivos reales, cuántos encontró el modelo.
- **Brier score:** mide el error de las probabilidades; un valor menor es mejor.

No usaremos el porcentaje total de aciertos como métrica principal porque T3 y T4 tienen pocos positivos. Un modelo que siempre diga 'no' tendría un porcentaje alto, pero no ayudaría al triaje.

In [ ]:
resumen_objetivos = {}
for periodo in ('fit', 'calibration', 'validation'):
    resumen_objetivos[periodo] = {}
    for objetivo, valores in evaluacion['targets'][periodo]['no_shared_text'].items():
        resumen_objetivos[periodo][objetivo] = {
            clave: valor
            for clave, valor in valores.items()
            if clave != 'class_counts'
        }
resumen_objetivos

## 8. Variables permitidas

La primera comparación usará narrativa normalizada y producto canónico. Empresa, estado y fecha serán candidatos: solo se añadirán si demuestran valor.

Se excluyen `Issue`, respuestas de la empresa y resultados T1–T4. Esas columnas revelarían la respuesta que el modelo intenta predecir; usar esa información se llama **fuga de información**.

In [ ]:
evaluacion['features']

## Siguiente paso

M3 comparará referencias simples. Estas referencias indicarán cuánto valor debe aportar un modelo antes de justificar mayor complejidad.